# Logistic Regression — Phân loại khu vực cao điểm bằng PySpark MLlib (NYC)

**Dữ liệu:** Silver Delta Lake trên MinIO (`s3a://lakehouse/silver/citibike`) · Kỳ **202401 – 202412**

**Hệ thống:** Citi Bike (NYC) - Đồ án Big Data phân tán

> **Dự đoán liệu một khu vực (`area_id`) trong một giờ nhất định có rơi vào trạng thái cao điểm (`rush area`) hay không trực tiếp trên Data Lakehouse.**

| Nhãn | Ý nghĩa |
|---|---|
| `0` | Khu vực không ở trạng thái cao điểm |
| `1` | Khu vực ở trạng thái cao điểm |


In [ ]:
import sys
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.spark_session import create_spark
from src.utils.paths import silver_path
import src.rush_area_model_spark as rush_area_model_spark

from src.rush_area_model_spark import (
    build_rush_area_dataset_spark,
    split_summary_table_spark,
    panel_summary_table_spark,
    area_summary_table_spark,
    plot_area_folium_spark,
    train_logistic_rush_area_model_spark,
    metrics_table_spark,
    plot_logistic_metrics_spark,
    plot_confusion_matrix_spark,
    show_top_rush_probabilities_spark,
    plot_rush_probability_map_spark,
    choose_snapshot_hour_spark,
    TOP_AREAS,
    RUSH_QUANTILE,
    MIN_RUSH_THRESHOLD,
)

# Create local Spark session with Delta configuration inside Jupyter
spark = create_spark("citibike-jupyter-ml")

SILVER_DATA_PATH = silver_path()
SYSTEM_NAME = "Citi Bike (NYC)"
CITY_NAME = "New York City"


## Khối 1 · Đơn vị quan sát và `area_id`

Khu vực được tạo bằng ô lưới tọa độ 0.01° × 0.01° trong Spark SQL phân tán:

$$
\mathrm{area\_lat}=\frac{\lfloor 100\cdot \mathrm{start\_lat}\rfloor}{100},\qquad
\mathrm{area\_lng}=\frac{\lfloor 100\cdot \mathrm{start\_lng}\rfloor}{100}
$$

$$
\mathrm{area\_id}=\text{concat}(\mathrm{area\_lat},\;\_\;,\mathrm{area\_lng})
$$


## Khối 2 · Tạo panel `area × hour` và gán nhãn rush

Quy trình xử lý thuần túy trên Spark DataFrame phân tán, lưu trữ trực tiếp trên bộ nhớ cụm.

In [ ]:
RUSH_LABELED = build_rush_area_dataset_spark(
    spark,
    SILVER_DATA_PATH,
    SYSTEM_NAME,
    city_name=CITY_NAME,
    top_areas=TOP_AREAS,
    rush_quantile=RUSH_QUANTILE,
    min_threshold=MIN_RUSH_THRESHOLD,
)

print(f"Dataset Panel generated.")

## Khối 3 · Kiểm tra panel, khoảng train/test và bản đồ `area_id`

Thống kê dữ liệu huấn luyện và bản đồ phân bố nhu cầu theo khu vực.

In [ ]:
display(split_summary_table_spark(RUSH_LABELED))
display(panel_summary_table_spark(RUSH_LABELED))
display(area_summary_table_spark(RUSH_LABELED, top_n=10))


In [ ]:
plot_area_folium_spark(RUSH_LABELED, metric="total_demand")


## Khối 4 · PySpark MLlib Logistic Regression

Xây dựng Spark Pipeline tiền xử lý và huấn luyện phân loại Nhị phân.

In [ ]:
RUSH_RESULT = train_logistic_rush_area_model_spark(RUSH_LABELED)

display(metrics_table_spark(RUSH_RESULT))


## Khối 5 · Đánh giá mô hình học máy phân tán

Đánh giá các chỉ số Precision, Recall, F1, ROC-AUC, PR-AUC và ma trận nhầm lẫn.

In [ ]:
plot_logistic_metrics_spark(RUSH_RESULT)
plot_confusion_matrix_spark(RUSH_RESULT)

## Khối 6 · Dự đoán theo thời điểm do người dùng chọn

Hiển thị bản đồ dự báo nhiệt xác suất quá tải xe tại thời điểm test cụ thể.

In [ ]:
# TỰ NHẬP NGÀY/GIỜ BẠN MUỐN DỰ ĐOÁN TẠI ĐÂY (chọn giờ chẵn :00:00)
DEMO_HOUR = "2024-11-25 08:00:00" 

# (Hoặc nếu bạn muốn hệ thống tự động gợi ý một giờ chẵn ngẫu nhiên hợp lệ)
# SUGGESTED_HOUR = choose_snapshot_hour_spark(RUSH_RESULT)
# DEMO_HOUR = SUGGESTED_HOUR

print(f"Testing hour: {DEMO_HOUR}")

# Hiển thị bảng xếp hạng xác suất cao điểm
display(show_top_rush_probabilities_spark(RUSH_RESULT, timestamp_hour=DEMO_HOUR, top_n=15))

# Vẽ bản đồ nhiệt dự báo
plot_rush_probability_map_spark(RUSH_RESULT, timestamp_hour=DEMO_HOUR)
